# 02 — Reconstruct Execution Episodes

This notebook converts window-level observations into execution-level records.

A raw observation is one machine/time-window record. Multiple observations may belong to the same reconstructed execution. The final `execution_id` will be created here, not inherited directly from `collection_id + instance_index`.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate repository root
REPO_ROOT = Path.cwd()

while REPO_ROOT.name != "masters-project" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

if REPO_ROOT.name != "masters-project":
    raise RuntimeError(
        "Could not locate repository root named 'masters-project'."
    )

# Canonical observation-level dataset from notebook 01
OBSERVATIONS_PATH = (
    REPO_ROOT
    / "data"
    / "processed"
    / "google_subset_v1"
    / "google_subset_observations_v1.parquet"
)

if not OBSERVATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Canonical observations file not found:\n{OBSERVATIONS_PATH}"
    )

df = pd.read_parquet(OBSERVATIONS_PATH)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("Repository root:", REPO_ROOT)
print("Loaded:", OBSERVATIONS_PATH)
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Repository root: /Users/meghanakoti/masterproject/masters-project
Loaded: /Users/meghanakoti/masterproject/masters-project/data/processed/google_subset_v1/google_subset_observations_v1.parquet
Shape: (405894, 53)
Columns:
['Unnamed: 0', 'time', 'instance_events_type', 'collection_id', 'scheduling_class', 'collection_type', 'priority', 'alloc_collection_id', 'instance_index', 'machine_id', 'resource_request', 'constraint', 'collections_events_type', 'user', 'collection_name', 'collection_logical_name', 'start_after_collection_ids', 'vertical_scaling', 'scheduler', 'start_time', 'end_time', 'average_usage', 'maximum_usage', 'random_sample_usage', 'assigned_memory', 'page_cache_memory', 'cycles_per_instruction', 'memory_accesses_per_instruction', 'sample_rate', 'cpu_usage_distribution', 'tail_cpu_usage_distribution', 'cluster', 'event', 'failed', 'raw_req_cpu', 'raw_req_memory', 'raw_avg_cpu', 'raw_avg_memory', 'raw_peak_cpu', 'raw_peak_memory', 'event_time_us', 'usage_start_time_us', 'us

,Unnamed: 0,time,instance_events_type,collection_id,scheduling_class,collection_type,priority,alloc_collection_id,instance_index,machine_id,resource_request,constraint,collections_events_type,user,collection_name,collection_logical_name,start_after_collection_ids,vertical_scaling,scheduler,start_time,end_time,average_usage,maximum_usage,random_sample_usage,assigned_memory,page_cache_memory,cycles_per_instruction,memory_accesses_per_instruction,sample_rate,cpu_usage_distribution,tail_cpu_usage_distribution,cluster,event,failed,raw_req_cpu,raw_req_memory,raw_avg_cpu,raw_avg_memory,raw_peak_cpu,raw_peak_memory,event_time_us,usage_start_time_us,usage_end_time_us,usage_window_duration_sec,requested_cpu_ncu,average_cpu_ncu,peak_cpu_ncu,requested_memory_normalized,average_memory_normalized,peak_memory_normalized,assigned_memory_normalized,page_cache_memory_normalized,observation_id
0,0,0,2,94591244395,3,1,200,0,144,168846390496,"{'cpus': 0.020660400390625, 'memory': 0.014434...",[],2,fn8Ve4Tdl/FVVvwXFGIKe4+Wo4zLjUL/557qdFVYu5M=,Hzsv/gF8CPQXdqpsfovDTC1TJNyphDxPu7vaTeNxA74=,YCuhYrnORLiUh9WGL5q5tkBevfwtucSnFr2qPZh6Kes=,[],1.0,0.0,274800000000,275100000000,"{'cpus': 0.00466156005859375, 'memory': 0.0059...","{'cpus': 0.01190185546875, 'memory': 0.0059356...","{'cpus': 0.0043487548828125, 'memory': None}",0.014435,0.000415,NaN,NaN,1.0,[0.00314331 0.00381088 0.00401306 0.00415039 0...,[0.00535583 0.00541687 0.00548553 0.00554657 0...,7,FAIL,1,0.020660,0.014435,0.004662,5.920410e-03,0.011902,5.935669e-03,0,274800000000,275100000000,300.0,0.020660,0.004662,0.011902,0.014435,5.920410e-03,5.935669e-03,0.014435,0.000415,94591244395::144::168846390496::274800000000::...
1,1,2517305308183,2,260697606809,2,0,360,221495397286,335,85515092,"{'cpus': 0.00724029541015625, 'memory': 0.0013...",[],2,DrrEIEWkWuW7RrZwpHLCN0k0A2J0usJeyt3wtqzZ7Kk=,hDGffcrF/rhQQEG8Uns/RMUK7R15DXjFnRasoKFhefI=,wcRcAMuop2OqH9EW4feH919tadFec5a11ply0hcS/C8=,[],2.0,0.0,1800713000000,1800714000000,"{'cpus': 0.0, 'memory': 9.5367431640625e-07}","{'cpus': 0.0, 'memory': 9.5367431640625e-07}","{'cpus': 0.0, 'memory': None}",0.000000,0.000000,NaN,NaN,1.0,[1.23977661e-05 1.23977661e-05 1.23977661e-05 ...,[1.23977661e-05 1.23977661e-05 1.23977661e-05 ...,7,FAIL,1,0.007240,0.001303,0.000000,9.536743e-07,0.000000,9.536743e-07,2517305308183,1800713000000,1800714000000,1.0,0.007240,0.000000,0.000000,0.001303,9.536743e-07,9.536743e-07,0.000000,0.000000,260697606809::335::85515092::1800713000000::18...
2,2,195684022913,6,276227177776,2,0,103,0,376,169321752432,"{'cpus': 0.048583984375, 'memory': 0.004165649...",[],6,/ivQBmewiFcXfGJdCUsEKx47NiRE29Tjiq3gw+zR2Cg=,kk6+maA6fvAdJ+VTU8AcpzQPTyVrx+ySt0MXRAyO8FU=,zCA2dl2PDptd82Hob906gE82JHzx0SbqA4mZurqZdmY=,[],2.0,1.0,81300000000,81600000000,"{'cpus': 0.024200439453125, 'memory': 0.002788...","{'cpus': 0.06005859375, 'memory': 0.0028457641...","{'cpus': 0.026458740234375, 'memory': None}",0.010422,0.000235,0.939919,0.001318,1.0,[0.01344299 0.01809692 0.0201416 0.02246094 0...,[0.02902222 0.02929688 0.0295105 0.0296936 0...,7,SCHEDULE,0,0.048584,0.004166,0.024200,2.788544e-03,0.060059,2.845764e-03,195684022913,81300000000,81600000000,300.0,0.048584,0.024200,0.060059,0.004166,2.788544e-03,2.845764e-03,0.010422,0.000235,276227177776::376::169321752432::81300000000::...
3,3,0,2,10507389885,3,0,200,0,1977,178294817221,"{'cpus': 0.0704345703125, 'memory': 0.04162597...",[],2,8qRmTJas/6XEBaA0l4Wt1+/qSLgc6p7u7JzoMSuT/M8=,fypwFjdqaQPSxCfeqVPCBAvFcnntmkRpQxwQ/vJsCxU=,uNjMQD1+DL9IgCFckx8lHOsCbyvLgKmZCmRjiyWZNhk=,[],2.0,0.0,1075500000000,1075800000000,"{'cpus': 0.047607421875, 'memory': 0.034423828...","{'cpus': 0.13330078125, 'memory': 0.03466796875}","{'cpus': 0.05084228515625, 'memory': None}",0.041626,0.000225,1.359102,0.007643,1.0,[0.03704834 0.04125977 0.04290771 0.04425049 0...,[0.05535889 0.05584717 0.05633545 0.05718994 0...,8,FAIL,1,0.070435,0.041626,0.047607,3.442383e-02,0.133301,3.466797e-02,0,1075500000000,1075800000000,300.0,0.070435,0.047607,0.1333

## Step 2 — Validate Reconstruction Inputs

Before reconstructing executions, verify that every observation has the identifiers and timestamp fields required for grouping.

`observation_id` uniquely identifies one raw machine/time-window observation. It is not the final execution identifier.

In [4]:
#Validate reconstruction input
required_columns = [
    "observation_id",
    "collection_id",
    "instance_index",
    "machine_id",
    "event_time_us",
    "usage_start_time_us",
    "usage_end_time_us",
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "The canonical observation dataset is missing:\n"
        + "\n".join(f"- {column}" for column in missing_columns)
    )

print("All required reconstruction columns are present.")
print("Total observations:", len(df))
print("Unique observation IDs:", df["observation_id"].nunique())
print(
    "Duplicate observation IDs:",
    df["observation_id"].duplicated().sum()
)

All required reconstruction columns are present.
Total observations: 405894
Unique observation IDs: 405894
Duplicate observation IDs: 0


## Step 3 — Prepare Canonical Timestamp Fields

Convert the canonical timestamp columns to numeric microsecond values and validate the usage intervals.

The original timestamp columns remain preserved. The derived `usage_duration_us` field represents the duration of an observation window, not necessarily the complete execution runtime.

In [ ]:
timestamp_columns = [
    "event_time_us",
    "usage_start_time_us",
    "usage_end_time_us",
]

for column in timestamp_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

invalid_timestamp_counts = df[timestamp_columns].isna().sum()

print("Missing canonical timestamps:")
display(invalid_timestamp_counts)

# Keep only rows with a valid usage interval
df = df.dropna(
    subset=["usage_start_time_us", "usage_end_time_us"]
).copy()

# Ensure interval ordering
df["usage_duration_us"] = (
    df["usage_end_time_us"] -
    df["usage_start_time_us"]
)

negative_duration_count = (
    df["usage_duration_us"] < 0
).sum()

print("Negative usage durations:", negative_duration_count)

if negative_duration_count > 0:
    raise ValueError(
        "Negative usage intervals detected. "
        "Inspect timestamps before reconstruction."
    )

df = df.sort_values(
    [
        "collection_id",
        "instance_index",
        "usage_start_time_us",
        "usage_end_time_us",
        "machine_id",
    ]
).reset_index(drop=True)

print("Prepared observations:", len(df))
display(
    df[
        [
            "observation_id",
            "collection_id",
            "instance_index",
            "machine_id",
            "event_time_us",
            "usage_start_time_us",
            "usage_end_time_us",
            "usage_duration_us",
        ]
    ].head()
)

## Step 4 — Audit Candidate Instance Groups

Use `(collection_id, instance_index)` as a candidate grouping key for inspection only.

This audit checks how many observations and machines appear in each candidate group and how wide its time span is. The results will determine whether additional episode separation is required before creating the final `execution_id`.

In [5]:
# This is only an audit key.
# It is not yet the final execution_id.
df["candidate_instance_key"] = (
    df["collection_id"].astype(str)
    + "||"
    + df["instance_index"].astype(str)
)

instance_audit = (
    df.groupby("candidate_instance_key")
      .agg(
          observation_count=("observation_id", "nunique"),
          machine_count=("machine_id", "nunique"),
          first_usage_start_us=("usage_start_time_us", "min"),
          last_usage_end_us=("usage_end_time_us", "max"),
      )
      .reset_index()
)

instance_audit["span_us"] = (
    instance_audit["last_usage_end_us"]
    - instance_audit["first_usage_start_us"]
)

print(
    "Unique candidate collection-instance groups:",
    len(instance_audit)
)

print("\nObservations per candidate group:")
display(instance_audit["observation_count"].describe())

print("\nMachines per candidate group:")
display(instance_audit["machine_count"].describe())

print("\nCandidate groups with multiple machines:")
display(
    instance_audit[
        instance_audit["machine_count"] > 1
    ].head(10)
)

print("\nLargest candidate groups:")
display(
    instance_audit.sort_values(
        "observation_count",
        ascending=False
    ).head(10)
)

Unique candidate collection-instance groups: 242946

Observations per candidate group:


count    242946.000000
mean          1.670717
std          17.788001
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max        7704.000000
Name: observation_count, dtype: float64


Machines per candidate group:


count    242946.000000
mean          1.620002
std          15.906318
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max        6640.000000
Name: machine_count, dtype: float64


Candidate groups with multiple machines:


,candidate_instance_key,observation_count,machine_count,first_usage_start_us,last_usage_end_us,span_us
103,10121672||1303,2,2,474900000000,475200000000,300000000
116,10121672||1470,2,2,474900000000,475200000000,300000000
163,10121672||2050,2,2,474900000000,475200000000,300000000
175,10121672||2220,2,2,474900000000,475200000000,300000000
196,10121672||2447,2,2,474900000000,475200000000,300000000
203,10121672||2516,2,2,474900000000,475200000000,300000000
237,10121672||738,2,2,474900000000,2031600000000,1556700000000
247,10121672||924,2,2,474900000000,2031600000000,1556700000000
559,10445557720||1850,2,2,375600000000,2670000000000,2294400000000
800,105142593258||1055,13,13,678000000000,678300000000,300000000



Largest candidate groups:


,candidate_instance_key,observation_count,machine_count,first_usage_start_us,last_usage_end_us,span_us
1927,116646838687||17,7704,6640,361200000000,2288923000000,1927723000000
7656,14423004927||969,1430,1430,1979700000000,1980000000000,300000000
2124,122697038507||1191,1308,1308,1944000000000,1944300000000,300000000
127632,348020181306||5,1135,1135,2544777000000,2544778000000,1000000
7580,14423004927||1021,994,994,1979700000000,1980000000000,300000000
49308,260725669813||176,905,903,25800000000,1175400000000,1149600000000
49234,260725669813||112,767,758,25800000000,1175400000000,1149600000000
2270,122697038507||5012,686,686,1944000000000,1944300000000,300000000
3770,128073966478||2432,587,586,115800000000,430500000000,314700000000
49241,260725669813||118,580,578,25800000000,1175400000000,1149600000000


In [6]:
# Sort observations chronologically within each candidate group
gap_df = df.sort_values(
    [
        "candidate_instance_key",
        "usage_start_time_us",
        "usage_end_time_us",
    ]
).copy()

# Previous observation end within the same candidate group
gap_df["previous_usage_end_us"] = (
    gap_df
    .groupby("candidate_instance_key")["usage_end_time_us"]
    .shift(1)
)

# Time gap between consecutive observations
gap_df["gap_from_previous_us"] = (
    gap_df["usage_start_time_us"]
    - gap_df["previous_usage_end_us"]
)

# Exclude the first observation in each group
positive_gaps = gap_df.loc[
    gap_df["gap_from_previous_us"].notna()
    & (gap_df["gap_from_previous_us"] > 0),
    "gap_from_previous_us",
]

print("Observations with a previous observation:", gap_df["previous_usage_end_us"].notna().sum())
print("Positive temporal gaps:", len(positive_gaps))

print("\nPositive gap distribution in microseconds:")
display(
    positive_gaps.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )
)

print("\nLargest temporal gaps:")
display(
    gap_df[
        gap_df["gap_from_previous_us"].notna()
    ]
    .sort_values("gap_from_previous_us", ascending=False)
    [
        [
            "candidate_instance_key",
            "observation_id",
            "machine_id",
            "previous_usage_end_us",
            "usage_start_time_us",
            "gap_from_previous_us",
        ]
    ]
    .head(20)
)

Observations with a previous observation: 162948
Positive temporal gaps: 10639

Positive gap distribution in microseconds:


count                 10639.0
mean      599492682018.986694
std       585178365982.275879
min              1500000000.0
50%            346200000000.0
75%           1047300000000.0
90%           1521900000000.0
95%           1774013000000.0
99%      2222692340000.015625
99.9%         2400000000000.0
max           2530500000000.0
Name: gap_from_previous_us, dtype: Float64


Largest temporal gaps:


,candidate_instance_key,observation_id,machine_id,previous_usage_end_us,usage_start_time_us,gap_from_previous_us
247628,96110849886||4625,96110849886::4625::2272276147::2537100000000::...,2272276147,6600000000,2537100000000,2530500000000
108145,134216236214||474,134216236214::474::225350648716::2561700000000...,225350648716,138900000000,2561700000000,2422800000000
250902,134216236214||507,134216236214::507::52214052853::2561700000000:...,52214052853,138900000000,2561700000000,2422800000000
87593,330587274558||1269,330587274558::1269::1377139849::2677800000000:...,1377139849,277800000000,2677800000000,2400000000000
154789,330587274558||1919,330587274558::1919::346992050627::267780000000...,346992050627,277800000000,2677800000000,2400000000000
374497,330587274558||2267,330587274558::2267::91986151568::2677800000000...,91986151568,277800000000,2677800000000,2400000000000
80856,330587274558||38,330587274558::38::375996823156::2677800000000:...,375996823156,277800000000,2677800000000,2400000000000
263174,330587274558||383,330587274558::383::1377328837::2677800000000::...,1377328837,277800000000,2677800000000,2400000000000
106045,330587274558||411,330587274558::411::21532187::2677800000000::26...,21532187,277800000000,2677800000000,2400000000000
230759,330587274558||629,330587274558::629::22672367::2677800000000::26...,22672367,277800000000,2677800000000,2400000000000


## Step 6 — Interpret Temporal Gaps

Convert the temporal gaps from microseconds into minutes and days.

This helps distinguish normal spacing between usage windows from larger inactive periods that may indicate a new execution episode.

In [7]:
gap_summary = gap_df[
    gap_df["gap_from_previous_us"].notna()
    & (gap_df["gap_from_previous_us"] > 0)
].copy()

gap_summary["gap_minutes"] = (
    gap_summary["gap_from_previous_us"] / 60_000_000
)

gap_summary["gap_days"] = (
    gap_summary["gap_from_previous_us"] / 86_400_000_000
)

print("Positive gap summary in minutes:")
display(
    gap_summary["gap_minutes"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )
)

print("Positive gap summary in days:")
display(
    gap_summary["gap_days"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )
)

print("Most common positive gaps:")
display(
    gap_summary["gap_minutes"]
    .round(3)
    .value_counts()
    .head(20)
)

Positive gap summary in minutes:


count         10639.0
mean        9991.5447
std       9752.972766
min              25.0
50%            5770.0
75%           17455.0
90%           25365.0
95%      29566.883333
99%      37044.872333
99.9%         40000.0
max           42175.0
Name: gap_minutes, dtype: Float64

Positive gap summary in days:


count      10639.0
mean      6.938573
std       6.772898
min       0.017361
50%       4.006944
75%      12.121528
90%      17.614583
95%      20.532558
99%      25.725606
99.9%    27.777778
max      29.288194
Name: gap_days, dtype: Float64

Most common positive gaps:


gap_minutes
2420.0      1528
19540.0      598
1660.0       441
18965.0      276
25365.0      215
4206.85      170
15720.0      167
1790.0       156
1675.567     121
17455.0      117
940.0        117
15681.95     113
3256.683     112
20335.0      103
33.083       102
10245.0      102
1540.0        90
545.0         88
6185.0        88
9962.033      85
Name: count, dtype: Int64

Above:
The smallest positive gap is 25 minutes.
The median positive gap is approximately 4 days.
Most positive gaps represent long inactive periods, not normal adjacent observation windows.
Therefore, we can use a 5-minute continuity threshold, matching the approximate observation-window duration.

Any gap greater than 5 minutes will start a new execution episode.

## Step 7 — Define Execution Episode Boundaries

The usage observations represent approximately five-minute windows.

The smallest positive gap between observations is 25 minutes, which is already larger than one normal observation window. Therefore, a gap greater than five minutes is treated as a boundary between execution episodes.

A machine change alone does not create a new episode. The episode boundary is based on a temporal gap within the same candidate collection-instance group.

In [9]:
# Approximate observation-window duration
OBSERVATION_WINDOW_US = 5 * 60 * 1_000_000  # five minutes

# Validate the observed positive gaps
minimum_positive_gap_us = (
    gap_df.loc[
        gap_df["gap_from_previous_us"].notna()
        & (gap_df["gap_from_previous_us"] > 0),
        "gap_from_previous_us"
    ].min()
)

print(
    "Minimum positive gap in minutes:",
    minimum_positive_gap_us / 60_000_000
)

if minimum_positive_gap_us <= OBSERVATION_WINDOW_US:
    print(
        "Warning: some positive gaps are within one observation window."
    )
else:
    print(
        "All positive gaps exceed the five-minute observation window."
    )

# A new episode begins after a gap larger than one observation window.
gap_df["new_episode"] = (
    gap_df["gap_from_previous_us"] > OBSERVATION_WINDOW_US
)

# The first observation in each candidate group starts episode zero.
gap_df["new_episode"] = (
    gap_df["new_episode"]
    | gap_df["previous_usage_end_us"].isna()
)

# Number episodes chronologically within each candidate group.
gap_df["episode_number"] = (
    gap_df
    .groupby("candidate_instance_key")["new_episode"]
    .cumsum()
    .astype(int)
)

# Create the final reconstructed execution ID.
gap_df["execution_id"] = (
    gap_df["candidate_instance_key"]
    + "||episode_"
    + gap_df["episode_number"].astype(str)
)

print("Reconstructed execution episodes:")
print("Unique execution IDs:", gap_df["execution_id"].nunique())
print("Total observations:", len(gap_df))

Minimum positive gap in minutes: 25.0
All positive gaps exceed the five-minute observation window.
Reconstructed execution episodes:
Unique execution IDs: 253585
Total observations: 405894


## Step 8 — Create the Reconstructed Execution ID

The dataset does not provide an explicit episode ID.

In the previous step, we inferred episodes by:

- grouping observations by `collection_id + instance_index`;
- ordering them by time;
- starting a new episode when the time gap was greater than five minutes.

This cell creates the final `execution_id` by combining the candidate group with its episode number.

For example:

```text
candidate group: 100||7
episode 0:       100||7||episode_0
episode 1:       100||7||episode_1

In [11]:
# Reload the canonical observation dataset to remove
# any partially merged columns from the previous attempt.
base_df = pd.read_parquet(OBSERVATIONS_PATH)

# Remove older or previously generated reconstruction columns.
base_df = base_df.drop(
    columns=[
        "execution_id",
        "candidate_instance_key",
        "episode_number",
    ],
    errors="ignore",
)

# Create a clean lookup table from the temporal reconstruction.
execution_lookup = (
    gap_df[
        [
            "observation_id",
            "candidate_instance_key",
            "episode_number",
            "execution_id",
        ]
    ]
    .drop_duplicates("observation_id")
)

# Join reconstructed execution information back to each observation.
df = base_df.merge(
    execution_lookup,
    on="observation_id",
    how="left",
    validate="one_to_one",
)

# Validate the assignment.
print("Execution ID assignment complete.")
print("Total observations:", len(df))
print(
    "Missing reconstructed execution IDs:",
    df["execution_id"].isna().sum()
)
print(
    "Unique reconstructed executions:",
    df["execution_id"].nunique()
)

display(
    df[
        [
            "observation_id",
            "candidate_instance_key",
            "episode_number",
            "execution_id",
        ]
    ].head()
)

Execution ID assignment complete.
Total observations: 405894
Missing reconstructed execution IDs: 0
Unique reconstructed executions: 253585


,observation_id,candidate_instance_key,episode_number,execution_id
0,94591244395::144::168846390496::274800000000::...,94591244395||144,1,94591244395||144||episode_1
1,260697606809::335::85515092::1800713000000::18...,260697606809||335,1,260697606809||335||episode_1
2,276227177776::376::169321752432::81300000000::...,276227177776||376,1,276227177776||376||episode_1
3,10507389885::1977::178294817221::1075500000000...,10507389885||1977,1,10507389885||1977||episode_1
4,25911621841::3907::231364893292::1565315000000...,25911621841||3907,1,25911621841||3907||episode_1


## Step 9 — Aggregate Observations Into Execution Records

The observations now have reconstructed execution IDs and normalized resource fields.

This step creates one row per execution and calculates:

- execution start and end time;
- runtime;
- number of observations;
- number of machines;
- requested CPU and memory;
- average CPU and memory usage;
- peak CPU and memory usage.

Peak usage is calculated using the maximum observed value within each reconstructed execution.

In [ ]:
# Verify required columns before aggregation.
required_columns = [
    "execution_id",
    "collection_id",
    "instance_index",
    "candidate_instance_key",
    "episode_number",
    "observation_id",
    "machine_id",
    "usage_start_time_us",
    "usage_end_time_us",
    "requested_cpu_ncu",
    "requested_memory_normalized",
    "average_cpu_ncu",
    "average_memory_normalized",
    "peak_cpu_ncu",
    "peak_memory_normalized",
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Create one row per reconstructed execution.
execution_df = (
    df.groupby("execution_id", as_index=False)
      .agg(
          collection_id=("collection_id", "first"),
          instance_index=("instance_index", "first"),
          candidate_instance_key=("candidate_instance_key", "first"),
          episode_number=("episode_number", "first"),

          execution_start_time_us=(
              "usage_start_time_us",
              "min",
          ),
          execution_end_time_us=(
              "usage_end_time_us",
              "max",
          ),

          observation_count=("observation_id", "nunique"),
          machine_count=("machine_id", "nunique"),

          requested_cpu_ncu=("requested_cpu_ncu", "first"),
          requested_memory_normalized=(
              "requested_memory_normalized",
              "first",
          ),

          average_cpu_ncu=("average_cpu_ncu", "mean"),
          average_memory_normalized=(
              "average_memory_normalized",
              "mean",
          ),

          peak_cpu_ncu=("peak_cpu_ncu", "max"),
          peak_memory_normalized=(
              "peak_memory_normalized",
              "max",
          ),
      )
)

# Calculate runtime.
execution_df["runtime_us"] = (
    execution_df["execution_end_time_us"]
    - execution_df["execution_start_time_us"]
)

execution_df["runtime_sec"] = (
    execution_df["runtime_us"] / 1_000_000
)

# Calculate resource slack.
execution_df["slack_cpu_ncu"] = (
    execution_df["requested_cpu_ncu"]
    - execution_df["peak_cpu_ncu"]
)

execution_df["slack_memory_normalized"] = (
    execution_df["requested_memory_normalized"]
    - execution_df["peak_memory_normalized"]
)

print("Execution-level dataset created.")
print("Execution records:", len(execution_df))
print(
    "Unique execution IDs:",
    execution_df["execution_id"].nunique()
)

display(execution_df.head())

Episode-number distribution:


,episode_number,execution_count
0,1,242946
1,2,10291
2,3,340
3,4,8


Largest reconstructed executions by observation count:


,execution_id,candidate_instance_key,episode_number,execution_start_time_us,execution_end_time_us,observation_count,machine_count,runtime_sec,requested_cpu_ncu,requested_memory_normalized,peak_cpu_ncu,peak_memory_normalized
1947,116646838687||17||episode_2,116646838687||17,2,2288922000000,2288923000000,6566,6566,1.0,0.001137,0.006508,0.000000,9.536743e-07
8134,14423004927||969||episode_1,14423004927||969,1,1979700000000,1980000000000,1430,1430,300.0,0.001060,0.000197,0.000647,1.029968e-04
2155,122697038507||1191||episode_1,122697038507||1191,1,1944000000000,1944300000000,1308,1308,300.0,0.013214,0.005211,0.025024,3.471375e-03
1946,116646838687||17||episode_1,116646838687||17,1,361200000000,361217000000,1138,1138,17.0,0.001137,0.006508,0.005554,7.591248e-04
134845,348020181306||5||episode_1,348020181306||5,1,2544777000000,2544778000000,1135,1135,1.0,0.000000,0.013016,0.000000,9.536743e-07
8056,14423004927||1021||episode_1,14423004927||1021,1,1979700000000,1980000000000,994,994,300.0,0.001171,0.000196,0.000647,1.029968e-04
53392,260725669813||176||episode_1,260725669813||176,1,25800000000,26100000000,903,903,300.0,0.009491,0.001303,0.001392,5.294800e-03
53271,260725669813||112||episode_1,260725669813||112,1,25800000000,26100000000,758,758,300.0,0.011154,0.001303,0.001392,5.294800e-03
2303,122697038507||5012||episode_1,122697038507||5012,1,1944000000000,1944300000000,686,686,300.0,0.013214,0.005211,0.025024,3.471375e-03
3896,128073966478||2432||episode_1,128073966478||2432,1,115800000000,116041000000,586,586,241.0,0.002602,0.000228,0.004860,1.831055e-04


Execution-size summary:


,observation_count,machine_count,runtime_sec
count,253585.000000,253585.000000,253585.0
mean,1.600623,1.600595,225.343475
std,15.606338,15.606330,122.660864
min,1.000000,1.000000,1.0
25%,1.000000,1.000000,105.0
50%,1.000000,1.000000,300.0
75%,1.000000,1.000000,300.0
max,6566.000000,6566.000000,300.0


Total execution records: 253585
Unique execution IDs: 253585
Duplicate execution IDs: 0


In [15]:
print("Unique execution IDs:",
      execution_df["execution_id"].nunique())

print("Duplicate execution IDs:",
      execution_df["execution_id"].duplicated().sum())

print("Negative runtimes:",
      (execution_df["runtime_us"] < 0).sum())

print("Missing runtimes:",
      execution_df["runtime_sec"].isna().sum())

print("\nRuntime summary in seconds:")
display(execution_df["runtime_sec"].describe())

print("\nObservations per execution:")
display(execution_df["observation_count"].describe())

print("\nMachines per execution:")
display(execution_df["machine_count"].describe())

Unique execution IDs: 253585
Duplicate execution IDs: 0
Negative runtimes: 0
Missing runtimes: 0

Runtime summary in seconds:


count      253585.0
mean     225.343475
std      122.660864
min             1.0
25%           105.0
50%           300.0
75%           300.0
max           300.0
Name: runtime_sec, dtype: Float64


Observations per execution:


count    253585.000000
mean          1.600623
std          15.606338
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max        6566.000000
Name: observation_count, dtype: float64


Machines per execution:


count    253585.000000
mean          1.600595
std          15.606330
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max        6566.000000
Name: machine_count, dtype: float64